# 58. The Solar Orbiter Heliospheric Imager (SoloHI) — Implementation / 구현

**Paper**: Howard, R. A. et al., "The Solar Orbiter Heliospheric Imager (SoloHI)", *A&A* 642, A13 (2020). DOI: 10.1051/0004-6361/201935202

This notebook implements four self-contained simulations of SoloHI's core engineering and science concepts:

1. **APS-CMOS detector noise model** — 5T pinned-photodiode pixel: read noise, dark current, photon shot noise, full-well saturation. CDS-corrected response curve at high/low gain.
2. **Heliospheric FOV coverage 5°–45°** — projection of SoloHI's 40°×40° field of view from 0.28 AU and 0.88 AU vantage points onto the inner heliosphere.
3. **Stray-light baffle suppression** — multi-stage forward-baffle diffraction attenuation as a function of solar elongation, reaching 10⁻¹³ B☉ at the outer FOV.
4. **Out-of-ecliptic 3D Thomson-scattering reconstruction** — toy tomographic recovery of an axisymmetric streamer from line-of-sight integrals at multiple ecliptic-inclination angles.

이 노트북은 SoloHI의 핵심 엔지니어링·과학 개념 네 가지를 자족적인 시뮬레이션으로 구현한다.
1. **APS-CMOS 검출기 잡음 모델** — 5T pinned-photodiode 픽셀의 read noise, dark current, photon shot noise, full-well 포화. high/low gain CDS 보정 응답 곡선.
2. **5°–45° 헬리오스피어 시야 커버리지** — 0.28 AU 및 0.88 AU 시점에서 SoloHI의 40°×40° 시야를 내부 헬리오스피어에 투영.
3. **잡광 baffle 억제** — 다단 forward baffle의 회절 감쇠를 태양 elongation 함수로, outer FOV에서 10⁻¹³ B☉ 수준까지.
4. **황도면 외 3D Thomson-산란 재구성** — 여러 황도경사각의 LOS 적분으로부터 축대칭 streamer를 토모그래피적으로 복원하는 toy 시뮬레이션.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from typing import Callable, Tuple, Optional

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# Physical constants
R_SUN_KM = 6.957e5  # km
AU_KM = 1.495978707e8  # km
AU_RSUN = AU_KM / R_SUN_KM  # ~215 R_sun per AU
SIGMA_T_CM2 = 6.6524587e-25  # Thomson scattering cross section (cm^2)
B_SUN = 1.0  # Mean solar disk brightness, in units of B_sun (for relative work)

# SoloHI key instrument parameters (Table 1 of Howard et al. 2020)
SOLOHI_FOCAL_MM = 55.9  # mm
SOLOHI_PIXEL_UM = 10.0  # um
SOLOHI_BORESIGHT_DEG = 25.0
SOLOHI_INNER_DEG = 5.4
SOLOHI_OUTER_DEG = 44.9
SOLOHI_DETECTOR_PIX = 3968  # 4-die mosaic
SOLOHI_QE = 0.32  # 480-750 nm average
SOLOHI_FULL_WELL_LOW_E = 86_400
SOLOHI_FULL_WELL_HIGH_E = 19_200
SOLOHI_READ_NOISE_LOW_E = 35.1
SOLOHI_READ_NOISE_HIGH_E = 5.8
SOLOHI_DARK_BOL_E_PIX_S = 0.3
SOLOHI_DARK_EOL_E_PIX_S = 2.0
SOLOHI_BANDPASS_NM = (500.0, 850.0)

## Part 1: APS-CMOS Detector Noise Model / APS-CMOS 검출기 잡음 모델

SoloHI uses a custom 4-die CMOS APS in pinwheel arrangement (3968×3968 px, 10 µm pitch). Each pixel is a 5T pinned-photodiode with on-chip Correlated Double Sampling (CDS), MIM-capacitor-selectable gain, and progressive-scan rolling-curtain shutter.

The total pixel signal in electrons is

$$N_{\rm signal}(t) = \eta\,\Phi_\gamma\,t \;+\; D(T)\,t \;+\; N_{\rm read}\;+\;\xi_{\rm Poisson}$$

with QE $\eta = 0.32$, dark current $D(T)$ depending strongly on detector temperature, read noise (5.8 e⁻ high-gain / 35.1 e⁻ low-gain), and Poisson photon noise. The signal saturates at the gain-dependent full well.

We implement the detector model and verify three reference values from §4.6 of the paper:
- Read-noise histogram median (high gain) ≈ 5.7 e⁻
- Linearity: ±1 % over most of the range, ±5 % only in curved tails (Fig. 21 right)
- Dynamic range $D = N_{\rm full\;well}/N_{\rm read}$ ≈ 86 400/35.1 ≈ 2 463 (low gain)

SoloHI는 4-die pinwheel APS 모자이크(3968×3968, 10 µm pitch)를 사용한다. 각 픽셀은 5T pinned-photodiode + 온칩 CDS + MIM-capacitor 가변 gain + progressive-scan rolling-curtain shutter 구조다. 위 식은 픽셀당 전자 수의 전체 신호를 나타내며, QE·dark current·read noise·Poisson photon noise의 네 성분으로 구성된다.

In [ ]:
@dataclass
class APSPixelModel:
    """SoloHI 5T pinned-photodiode APS pixel signal-and-noise model.

    Captures the photon-to-electron chain plus noise terms documented in §4.6
    of Howard et al. (2020). The model is intentionally per-pixel: spatial
    column-pattern noise and 1/f drift are absorbed into ``read_noise_e``
    after CDS, since CDS suppresses kTC and 1/f noise.

    Attributes:
        qe: Quantum efficiency averaged over 480-750 nm bandpass.
        full_well_e: Saturation level in electrons (depends on gain mode).
        read_noise_e: 1-sigma read noise after CDS, in electrons.
        dark_current_e_per_s: Pixel dark current at operating temperature.
        gain_mode: ``"high"`` or ``"low"`` (sets MIM-cap conversion gain).
        bias_e: Constant bias offset added to the signal.
    """

    qe: float = SOLOHI_QE
    full_well_e: float = SOLOHI_FULL_WELL_HIGH_E
    read_noise_e: float = SOLOHI_READ_NOISE_HIGH_E
    dark_current_e_per_s: float = SOLOHI_DARK_BOL_E_PIX_S
    gain_mode: str = "high"
    bias_e: float = 50.0

    @classmethod
    def low_gain(cls) -> "APSPixelModel":
        """Build a low-gain (high-full-well) pixel preset."""
        return cls(full_well_e=SOLOHI_FULL_WELL_LOW_E,
                   read_noise_e=SOLOHI_READ_NOISE_LOW_E,
                   gain_mode="low")

    def expose(self,
               photon_rate_per_s: np.ndarray,
               exposure_s: float,
               rng: Optional[np.random.Generator] = None) -> np.ndarray:
        """Simulate one exposure of an array of pixels.

        Args:
            photon_rate_per_s: Incident photon rate per pixel (photons/s).
            exposure_s: Integration time in seconds.
            rng: Random generator for stochastic noise terms.

        Returns:
            Measured signal in electrons after CDS, clipped at full well.
        """
        rng = rng if rng is not None else np.random.default_rng(0)
        # Convert photons -> photo-electrons and add Poisson shot noise
        n_photoelectrons_mean = self.qe * photon_rate_per_s * exposure_s
        n_photoelectrons = rng.poisson(np.maximum(n_photoelectrons_mean, 0.0)).astype(float)
        # Dark current is also Poisson distributed
        dark_e_mean = self.dark_current_e_per_s * exposure_s
        n_dark = rng.poisson(np.full_like(photon_rate_per_s, dark_e_mean, dtype=float))
        # CDS-corrected read noise: zero-mean Gaussian
        n_read = rng.normal(0.0, self.read_noise_e, size=photon_rate_per_s.shape)
        # Total signal with bias
        total = self.bias_e + n_photoelectrons + n_dark + n_read
        # Clip at full well (saturation)
        return np.minimum(total, self.full_well_e)

    @property
    def dynamic_range(self) -> float:
        """Detector dynamic range = full_well / read_noise (dimensionless)."""
        return self.full_well_e / self.read_noise_e


# Instantiate both gain modes
high_gain = APSPixelModel()
low_gain = APSPixelModel.low_gain()
print(f"High-gain dynamic range: {high_gain.dynamic_range:.0f}")
print(f"Low-gain  dynamic range: {low_gain.dynamic_range:.0f}")
print(f"Paper reports (Table 1): high gain ~ 19200/5.8 = {19200/5.8:.0f}, "
      f"low gain ~ 86400/35.1 = {86400/35.1:.0f}")

In [ ]:
rng = np.random.default_rng(58)

# (1) Read-noise histogram from a dark, zero-photon, very short exposure
n_pix = 64 * 64  # one of the 16 calibration regions used in Fig. 21
dark_pixels = np.zeros(n_pix)
dark_signal_high = high_gain.expose(dark_pixels, exposure_s=0.001, rng=rng) - high_gain.bias_e
dark_signal_low = low_gain.expose(dark_pixels, exposure_s=0.001, rng=rng) - low_gain.bias_e

# (2) Linearity sweep: vary photon rate, hold exposure fixed
photon_rates = np.logspace(0, 6, 80)  # 1 to 1e6 photons/s
exposure = 1.0
mean_signal_high = []
mean_signal_low = []
for rate in photon_rates:
    rate_arr = np.full(2048, rate)
    sig_h = high_gain.expose(rate_arr, exposure, rng=rng) - high_gain.bias_e
    sig_l = low_gain.expose(rate_arr, exposure, rng=rng) - low_gain.bias_e
    mean_signal_high.append(np.mean(sig_h))
    mean_signal_low.append(np.mean(sig_l))
mean_signal_high = np.array(mean_signal_high)
mean_signal_low = np.array(mean_signal_low)

# Ideal linear response for comparison
ideal_high = high_gain.qe * photon_rates * exposure
ideal_low = low_gain.qe * photon_rates * exposure

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: read noise histograms
axes[0].hist(dark_signal_high, bins=40, alpha=0.6, color='C0',
             label=f"High gain $\sigma$ = {np.std(dark_signal_high):.2f} e$^-$")
axes[0].hist(dark_signal_low, bins=40, alpha=0.6, color='C3',
             label=f"Low gain $\sigma$ = {np.std(dark_signal_low):.2f} e$^-$")
axes[0].set_xlabel("Pixel signal (e$^-$, bias-subtracted)")
axes[0].set_ylabel("Pixel count")
axes[0].set_title("Read-noise histogram (dark, zero-flux)\nFig. 21 left equivalent")
axes[0].legend()

# Right: linearity
axes[1].loglog(photon_rates, mean_signal_high, 'C0o-', ms=3, label='High gain (measured)')
axes[1].loglog(photon_rates, ideal_high, 'C0--', alpha=0.5, label='High gain (ideal QE-linear)')
axes[1].loglog(photon_rates, mean_signal_low, 'C3s-', ms=3, label='Low gain (measured)')
axes[1].loglog(photon_rates, ideal_low, 'C3--', alpha=0.5, label='Low gain (ideal QE-linear)')
axes[1].axhline(SOLOHI_FULL_WELL_HIGH_E, color='C0', ls=':', alpha=0.5, label='HG full well')
axes[1].axhline(SOLOHI_FULL_WELL_LOW_E, color='C3', ls=':', alpha=0.5, label='LG full well')
axes[1].set_xlabel(r"Photon rate (photons px$^{-1}$ s$^{-1}$)")
axes[1].set_ylabel(r"Mean signal (e$^-$)")
axes[1].set_title("Linearity sweep: full-well saturation (Fig. 21 right)")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

print(f"\nMeasured high-gain read noise: {np.std(dark_signal_high):.2f} e- "
      f"(paper: 5.7 median)")
print(f"Measured low-gain  read noise: {np.std(dark_signal_low):.2f} e- "
      f"(paper: 35.0 median)")

In [ ]:
# Dark-current temperature dependence (Arrhenius-like model).
# Empirical formula from CMOS APS literature: D(T) ~ D_0 * T^1.5 * exp(-Eg/(2 k T))
# We anchor to the paper's BOL value at -65 C goal and EOL at the same temperature.

K_BOLTZMANN_EVK = 8.617333e-5  # eV/K
SI_BANDGAP_EV = 1.115  # silicon at low T

def dark_current_model(temperature_K: np.ndarray, d_at_minus_65: float) -> np.ndarray:
    """Model dark current scaling with temperature.

    Args:
        temperature_K: Detector temperature in Kelvin.
        d_at_minus_65: Anchor dark-current value at -65 C in e-/pix/s.

    Returns:
        Dark current array in e-/pix/s.
    """
    t_anchor = 273.15 - 65.0  # K
    relative = (temperature_K / t_anchor)**1.5 * np.exp(
        -SI_BANDGAP_EV / (2.0 * K_BOLTZMANN_EVK) * (1.0 / temperature_K - 1.0 / t_anchor))
    return d_at_minus_65 * relative


temperature_C = np.linspace(-80, 0, 200)
temperature_K = temperature_C + 273.15
dark_bol = dark_current_model(temperature_K, SOLOHI_DARK_BOL_E_PIX_S)
dark_eol = dark_current_model(temperature_K, SOLOHI_DARK_EOL_E_PIX_S)

fig, ax = plt.subplots(figsize=(9, 5))
ax.semilogy(temperature_C, dark_bol, 'C0-', lw=2, label='BOL (anchor 0.3 e$^-$/px/s @ -65 C)')
ax.semilogy(temperature_C, dark_eol, 'C3-', lw=2, label='EOL (anchor 2.0 e$^-$/px/s @ -65 C)')
ax.axvline(-65.0, color='k', ls='--', alpha=0.5, label='Operating goal -65 C')
ax.axvline(-55.0, color='gray', ls=':', alpha=0.5, label='Required < -55 C')
ax.set_xlabel("Detector temperature (C)")
ax.set_ylabel(r"Dark current (e$^-$ px$^{-1}$ s$^{-1}$)")
ax.set_title("APS dark current vs. temperature (BOL & EOL)")
ax.legend()
plt.tight_layout()
plt.show()

# Compare dark vs read noise contributions for a 30 s nominal exposure
exposure_30 = 30.0
dark_e_30 = SOLOHI_DARK_EOL_E_PIX_S * exposure_30
read_e = SOLOHI_READ_NOISE_HIGH_E
print(f"30 s EOL dark contribution: {dark_e_30:.1f} e- (Poisson sigma {np.sqrt(dark_e_30):.2f} e-)")
print(f"High-gain read noise:       {read_e:.1f} e-")
print(f"Total noise floor (quad):   "
      f"{np.sqrt(dark_e_30 + read_e**2):.2f} e- per 30 s exposure")

## Part 2: Heliospheric FOV Coverage 5°–45° / 헬리오스피어 시야 5°–45° 커버리지

SoloHI's 40°×40° detector (48° corner-to-corner), with boresight 25° east of Sun centre, yields a science FOV of solar elongation $\varepsilon \in [5.4°, 44.9°]$ (Table 1). Because the spacecraft Sun-distance varies between 0.28 AU and ~0.88 AU, the *physical* range probed in heliocentric distance is dramatically different from STEREO/SECCHI HI-1 at 1 AU.

Given a spacecraft Sun-distance $d_{\rm sc}$, the projected radial distance corresponding to the Thomson-surface point at elongation $\varepsilon$ is

$$r_{\rm TS}(\varepsilon) = d_{\rm sc}\,\sin\varepsilon$$

(the closest-approach distance of the line of sight to the Sun). We project SoloHI's FOV onto the heliosphere from both perihelion (0.28 AU) and aphelion (0.88 AU), and compare with HI-1, HI-2, and LASCO C3.

SoloHI는 25° boresight + 40°×40° FOV로 elongation 5.4°–44.9°를 커버한다. 우주선 일심거리에 따라 동일 elongation이라도 물리 거리는 크게 달라지므로, 0.28 AU 근일점에서는 SoloHI가 사실상 inner-corona local imager가 된다. Thomson 표면 위 점의 일심 거리 $r_{\rm TS} = d_{\rm sc}\sin\varepsilon$를 사용해 FOV를 비교한다.

In [ ]:
def thomson_surface_radius(elongation_deg: np.ndarray,
                            spacecraft_AU: float) -> np.ndarray:
    """Heliocentric radius of the Thomson surface at given elongation.

    The Thomson surface is the sphere with the spacecraft-Sun line as
    diameter; the closest LOS approach to Sun centre at elongation ``eps``
    is ``d_sc * sin(eps)`` (Vourlidas & Howard 2006).

    Args:
        elongation_deg: Solar elongation in degrees.
        spacecraft_AU: Spacecraft Sun-distance in AU.

    Returns:
        Heliocentric distance in solar radii.
    """
    return spacecraft_AU * AU_RSUN * np.sin(np.deg2rad(elongation_deg))


# Instrument FOVs
elongation = np.linspace(0.5, 100, 500)
solohi_inner, solohi_outer = SOLOHI_INNER_DEG, SOLOHI_OUTER_DEG
hi1_inner, hi1_outer = 4.0, 24.0
hi2_inner, hi2_outer = 19.0, 89.0
lasco_c3_inner_rs, lasco_c3_outer_rs = 4.0, 30.0  # already in R_sun

distances = {
    "SoloHI @ 0.28 AU (perihelion)": 0.28,
    "SoloHI @ 0.5 AU": 0.5,
    "SoloHI @ 0.88 AU (max)": 0.88,
    "STEREO HI-1 @ 1 AU": 1.0,
    "STEREO HI-2 @ 1 AU": 1.0,
}

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: r_TS vs elongation
ax = axes[0]
for label, d in distances.items():
    r = thomson_surface_radius(elongation, d)
    ax.plot(elongation, r, lw=2 if "SoloHI" in label else 1.2,
            ls="-" if "SoloHI" in label else "--", label=label)
ax.axvspan(solohi_inner, solohi_outer, alpha=0.15, color="C0", label="SoloHI FOV (5.4-44.9 deg)")
ax.axvspan(hi1_inner, hi1_outer, alpha=0.10, color="C2", label="HI-1 FOV")
ax.axvspan(hi2_inner, hi2_outer, alpha=0.05, color="C3", label="HI-2 FOV")
ax.set_xlabel("Solar elongation (deg)")
ax.set_ylabel("Thomson-surface radius (R$_\odot$)")
ax.set_title("FOV projection: $r_{\mathrm{TS}} = d_{\mathrm{sc}}\sin\varepsilon$")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(1, 100)
ax.set_ylim(1, 300)
ax.legend(fontsize=8, loc="upper left")

# Right: heliocentric coverage bar chart
ax = axes[1]
bands = [
    ("LASCO C3 @ 1 AU", lasco_c3_inner_rs, lasco_c3_outer_rs, "C5"),
    ("HI-1 @ 1 AU", thomson_surface_radius(hi1_inner, 1.0),
     thomson_surface_radius(hi1_outer, 1.0), "C2"),
    ("HI-2 @ 1 AU", thomson_surface_radius(hi2_inner, 1.0),
     thomson_surface_radius(hi2_outer, 1.0), "C3"),
    ("SoloHI @ 0.88 AU", thomson_surface_radius(solohi_inner, 0.88),
     thomson_surface_radius(solohi_outer, 0.88), "C0"),
    ("SoloHI @ 0.5 AU", thomson_surface_radius(solohi_inner, 0.5),
     thomson_surface_radius(solohi_outer, 0.5), "C0"),
    ("SoloHI @ 0.28 AU", thomson_surface_radius(solohi_inner, 0.28),
     thomson_surface_radius(solohi_outer, 0.28), "C0"),
]
for i, (name, r_in, r_out, color) in enumerate(bands):
    ax.barh(i, r_out - r_in, left=r_in, color=color, alpha=0.7, edgecolor="k")
    ax.text(r_out + 1, i, f"{r_in:.1f}-{r_out:.1f} R$_\odot$",
            va="center", fontsize=9)
ax.set_yticks(range(len(bands)))
ax.set_yticklabels([b[0] for b in bands])
ax.set_xlabel("Heliocentric distance (R$_\odot$)")
ax.set_xscale("log")
ax.set_xlim(1, 500)
ax.set_title("Inner-heliosphere coverage by instrument")

plt.tight_layout()
plt.show()

# Quantify the inner-FOV advantage
r_solohi_inner_perihelion = thomson_surface_radius(solohi_inner, 0.28)
r_hi1_inner = thomson_surface_radius(hi1_inner, 1.0)
print(f"SoloHI @ 0.28 AU inner cutoff: {r_solohi_inner_perihelion:.1f} R_sun "
      f"({r_solohi_inner_perihelion / AU_RSUN * 1e3:.1f} m-AU)")
print(f"HI-1   @ 1.00 AU inner cutoff: {r_hi1_inner:.1f} R_sun")
print(f"Ratio: HI-1 / SoloHI = {r_hi1_inner / r_solohi_inner_perihelion:.2f}x "
      f"(SoloHI gets that much closer to Sun)")

## Part 3: Stray-Light Baffle Suppression vs. Solar Elongation / Baffle 잡광 억제 vs. Elongation

The K-corona signal is $10^{-9}$–$10^{-13}\,B_\odot$, so SoloHI must reject solar diffraction by ≥ 12 orders of magnitude. The architecture:

1. **Forward baffles** — 5 stages (HS edge → F1 → F2 → F3 → F4 + I0 trap). Each stage gives ~3 orders of diffraction attenuation (Young 1892 / Fraunhofer-edge model: $I_{\rm diff}(\theta) \propto \lambda / (\pi^2 \theta^2 \,L_{\rm edge})$ for a knife edge at angle $\theta$ off the shadow line).
2. **Interior baffles** — 9 baffles oriented to a light trap; double-bounce required, so reflectance squared.
3. **Aperture light-trap baffles** (AE1, AE2) — absorb residual diffraction over F4 + reflections off interior baffle tops.
4. **Peripheral baffle** — blocks direct view of the spacecraft heat shield.

We model the cumulative attenuation $A_N(\varepsilon) = \prod_{k=1}^{N} \alpha_k(\varepsilon)$ where each baffle contributes $\alpha_k \sim 10^{-3}$ at the inner FOV and improves with $\sin\varepsilon$. The requirement (Fig. 11): $1\times 10^{-10}\,B_\odot$ at inner FOV → $1\times 10^{-13}\,B_\odot$ at outer FOV.

K-corona 신호는 $10^{-9}$–$10^{-13}\,B_\odot$로, 다단 baffle 시스템을 통해 약 12 자릿수의 회절광 억제를 요구한다. forward baffle 5단(HS, F1, F2, F3, F4 + I0), interior baffle 9개, aperture light-trap 2단, peripheral baffle로 구성. 각 forward baffle은 약 $10^{-3}$ 회절 감쇠를 제공하며 5단 누적으로 외곽 FOV $10^{-13}\,B_\odot$ 수준 달성.

In [ ]:
def single_baffle_diffraction(elongation_deg: np.ndarray,
                                edge_distance_m: float = 0.683,
                                wavelength_m: float = 6.75e-7,
                                base_attenuation: float = 1.0e-3) -> np.ndarray:
    """Approximate per-stage diffraction attenuation for a knife-edge baffle.

    Uses a Fraunhofer-edge model: the diffracted intensity at angle theta
    off the geometric shadow line falls as ~1/(theta^2 * L), modulated by
    a base attenuation factor accounting for absorbing coating + finite edge.

    Args:
        elongation_deg: Solar elongation off the shadow line (degrees).
        edge_distance_m: Distance from edge to evaluation plane (m).
        wavelength_m: Wavelength used to evaluate diffraction (m).
        base_attenuation: Per-stage baseline attenuation factor.

    Returns:
        Attenuation factor per stage (dimensionless).
    """
    theta = np.deg2rad(np.maximum(elongation_deg, 1.0))  # avoid div zero
    # Knife-edge intensity ~ lambda / (4 pi^2 theta^2 L)
    diff_factor = wavelength_m / (4 * np.pi**2 * theta**2 * edge_distance_m)
    # Combined with base attenuation and clipped to physical [0, 1] range
    return np.clip(base_attenuation + diff_factor, 1e-15, 1.0)


def cumulative_baffle_attenuation(elongation_deg: np.ndarray,
                                    n_forward: int = 5,
                                    interior_reflectance: float = 0.05,
                                    n_interior_bounces: int = 2) -> dict:
    """Compute total stray-light attenuation across the SoloHI baffle system.

    Args:
        elongation_deg: Sample elongations (deg).
        n_forward: Number of forward baffle stages (HS, F1-F4 + I0 trap = 5).
        interior_reflectance: Per-bounce reflectance of interior baffle coating.
        n_interior_bounces: Number of bounces required in light trap.

    Returns:
        Dict containing per-stage and total attenuation arrays in B_sun units.
    """
    per_stage = single_baffle_diffraction(elongation_deg)
    forward_total = per_stage**n_forward
    interior_total = interior_reflectance**n_interior_bounces
    aperture_trap = 1.0e-2  # AE1, AE2 absorb residuals at ~1%
    peripheral = 1.0e-1  # planar baffle, 10% suppression of HS direct view
    total = forward_total * interior_total * aperture_trap * peripheral
    return {
        "per_stage": per_stage,
        "forward_total": forward_total,
        "interior_total": interior_total,
        "aperture_trap": aperture_trap,
        "peripheral": peripheral,
        "total": total,
    }


elongation = np.linspace(SOLOHI_INNER_DEG, SOLOHI_OUTER_DEG, 200)
result = cumulative_baffle_attenuation(elongation)

# Reference SoloHI requirement (Fig. 11 of Howard et al. 2020)
req_inner_b_sun = 1e-10
req_outer_b_sun = 1e-13
log_req = np.linspace(np.log10(req_inner_b_sun), np.log10(req_outer_b_sun), elongation.size)
requirement = 10.0**log_req

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Left: per-stage and cumulative attenuation
ax = axes[0]
ax.semilogy(elongation, result["per_stage"], 'k-', lw=1.5,
            label="Per-stage (single forward baffle)")
ax.semilogy(elongation, result["forward_total"], 'C0-', lw=2,
            label=f"5 forward stages = $\alpha^5$")
ax.semilogy(elongation, result["total"], 'C3-', lw=2,
            label="Total (forward + interior + trap + peripheral)")
ax.semilogy(elongation, requirement, 'k--', lw=2, alpha=0.7,
            label="Requirement (Fig. 11): $10^{-10}\to 10^{-13}\,B_\odot$")
ax.axvline(SOLOHI_INNER_DEG, color="gray", ls=":", alpha=0.5)
ax.axvline(SOLOHI_OUTER_DEG, color="gray", ls=":", alpha=0.5)
ax.set_xlabel("Solar elongation (deg)")
ax.set_ylabel("Attenuation (units of $B_\odot$)")
ax.set_title("Multi-stage baffle stray-light suppression")
ax.legend(fontsize=8)
ax.set_ylim(1e-16, 1e0)

# Right: contribution breakdown bar chart at inner FOV (5.4 deg)
ax = axes[1]
contribs = ["Forward x5", "Interior bounce x2", "Aperture trap", "Peripheral", "TOTAL"]
values_at_inner = [
    result["forward_total"][0],
    result["interior_total"],
    result["aperture_trap"],
    result["peripheral"],
    result["total"][0],
]
colors = ["C0", "C2", "C1", "C4", "C3"]
log_vals = -np.log10(values_at_inner)
bars = ax.barh(contribs, log_vals, color=colors)
for bar, val in zip(bars, values_at_inner):
    w = bar.get_width()
    ax.text(w + 0.3, bar.get_y() + bar.get_height()/2,
            f"{val:.1e}", va="center", fontsize=10)
ax.set_xlabel("Suppression (orders of magnitude)")
ax.set_title("Stray-light budget at inner FOV (5.4 deg)")
ax.set_xlim(0, max(log_vals) + 4)

plt.tight_layout()
plt.show()

print(f"Total attenuation @ 5.4 deg (inner): {result['total'][0]:.2e} B_sun")
print(f"Total attenuation @ 44.9 deg (outer): {result['total'][-1]:.2e} B_sun")
print(f"Requirement @ inner: {req_inner_b_sun:.0e}, @ outer: {req_outer_b_sun:.0e} B_sun")

## Part 4: Out-of-Ecliptic 3D Thomson-Scattering Reconstruction / 황도면 외 3D Thomson 산란 재구성

The Thomson-scattered K-corona brightness along a line of sight is

$$B(\mathbf{r}_{\rm sc}, \hat{\mathbf{p}}) = \int_{\rm LOS} G(\theta_{\rm sc}, \chi)\,n_e(\mathbf{r})\,ds$$

where $G$ is the Thomson geometric factor (Vourlidas & Howard 2006). With a single 1-AU vantage in the ecliptic (LASCO), recovering $n_e(\mathbf{r})$ requires forward-modelling assumptions because the LOS integrals sample a degenerate volume.

Solar Orbiter's >30° ecliptic-inclination orbit gives SoloHI multiple latitudinal vantages within ~2 weeks, breaking the degeneracy and enabling a tomographic inversion. We illustrate this by:

1. Building a synthetic axisymmetric streamer-like density profile $n_e(r, \theta_{\rm lat})$.
2. Computing LOS-integrated brightness from three vantages: ecliptic (0° lat), out-of-ecliptic-N (+30°), out-of-ecliptic-S (-30°).
3. Recovering the meridional density profile via simple least-squares inversion on the three vantage measurements.

Solar Orbiter의 >30° 황도경사 궤도는 ~2주 안에 남·북 위도를 sweep하여 SoloHI에 다중 vantage를 제공한다. 이는 단일 시점 LASCO에서는 불가능했던 tomographic inversion을 가능하게 한다. 본 cell에서는 축대칭 streamer 밀도 모델을 만들고, 세 vantage(0°, +30°, −30° lat)에서 LOS 적분을 계산한 뒤 최소제곱 역산으로 자오선 밀도를 복원한다.

In [ ]:
def axisymmetric_streamer_density(r_rsun: np.ndarray,
                                    lat_deg: np.ndarray,
                                    n_base: float = 5e7,
                                    h_streamer: float = 5.0,
                                    streamer_width_deg: float = 15.0) -> np.ndarray:
    """Synthetic axisymmetric streamer electron-density profile.

    Models a hydrostatic falloff with an enhanced equatorial belt
    (streamer) to mimic the heliospheric current sheet.

    Args:
        r_rsun: Heliocentric distance grid (R_sun).
        lat_deg: Heliographic latitude grid (deg).
        n_base: Reference density at 1 R_sun (cm^-3).
        h_streamer: Hydrostatic scale height (R_sun).
        streamer_width_deg: 1-sigma latitudinal width of the streamer belt.

    Returns:
        Density array of shape (len(r_rsun), len(lat_deg)) in cm^-3.
    """
    r_grid, lat_grid = np.meshgrid(r_rsun, lat_deg, indexing='ij')
    radial = n_base * np.exp(-(r_grid - 1.0) / h_streamer) / r_grid**2
    streamer_enhancement = 1.0 + 4.0 * np.exp(
        -0.5 * (lat_grid / streamer_width_deg)**2)
    return radial * streamer_enhancement


def thomson_geometric_factor(scatter_angle_deg: np.ndarray) -> np.ndarray:
    """Simplified Thomson-scattering geometric factor (Howard & Tappin 2009).

    Uses the symmetric-scattering approximation
    G(theta) = (1 + cos^2 theta) / 2, ignoring limb-darkening corrections.

    Args:
        scatter_angle_deg: Angle between Sun-scatterer and observer direction.

    Returns:
        Dimensionless geometric factor.
    """
    theta = np.deg2rad(scatter_angle_deg)
    return 0.5 * (1.0 + np.cos(theta)**2)


def los_brightness(spacecraft_lat_deg: float,
                    spacecraft_distance_AU: float,
                    elongation_deg: np.ndarray,
                    density_func: Callable,
                    n_los_steps: int = 100) -> np.ndarray:
    """Integrate Thomson-scattered brightness along LOS from a vantage point.

    The LOS originates at the spacecraft (r = d_sc, lat = sc_lat, lon = 0)
    and points at solar elongation epsilon in the meridional plane.
    We integrate the local Thomson contribution n_e * G * sigma_T along
    the LOS over a +/- 1.5 d_sc range.

    Args:
        spacecraft_lat_deg: Heliographic latitude of the spacecraft (deg).
        spacecraft_distance_AU: Sun-spacecraft distance (AU).
        elongation_deg: Sample LOS elongations (deg).
        density_func: Callable f(r_rsun, lat_deg) -> n_e cm^-3.
        n_los_steps: Number of integration points along each LOS.

    Returns:
        Brightness array B(elongation) in arbitrary K-corona units.
    """
    d_sc_rsun = spacecraft_distance_AU * AU_RSUN
    lat_sc = np.deg2rad(spacecraft_lat_deg)
    brightness = np.zeros_like(elongation_deg, dtype=float)
    s_array = np.linspace(0.0, 2.5 * d_sc_rsun, n_los_steps)  # LOS path length
    ds_rsun = s_array[1] - s_array[0]
    ds_cm = ds_rsun * R_SUN_KM * 1e5  # km -> cm

    for i, eps_deg in enumerate(elongation_deg):
        eps = np.deg2rad(eps_deg)
        # LOS endpoint coordinates as we step away from spacecraft toward Sun
        # In the meridional (Sun-spacecraft) plane:
        #   spacecraft at (d_sc * cos(lat_sc), 0, d_sc * sin(lat_sc))
        #   LOS direction: (-cos(eps), 0, +sin(eps)) -- toward Sun + above
        x_sc = d_sc_rsun * np.cos(lat_sc)
        z_sc = d_sc_rsun * np.sin(lat_sc)
        # LOS direction depends on whether we look above or below disk; use signed eps
        dx, dz = -np.cos(eps), np.sin(eps)
        x_los = x_sc + s_array * dx
        z_los = z_sc + s_array * dz
        r_los = np.sqrt(x_los**2 + z_los**2)
        # Latitude of each LOS point (deg)
        lat_los = np.rad2deg(np.arcsin(np.clip(z_los / np.maximum(r_los, 1e-3), -1, 1)))

        # Scattering angle: angle between Sun-scatterer and observer direction
        # Sun direction at scatterer: -(x_los, z_los)/r_los; observer direction = -LOS direction
        cos_scatter = -(x_los * dx + z_los * dz) / np.maximum(r_los, 1e-3)
        scatter_angle = np.rad2deg(np.arccos(np.clip(cos_scatter, -1, 1)))

        # Density along LOS (interpolate scalar by evaluating directly)
        n_e = np.array([density_func(np.array([r]), np.array([lat]))[0, 0]
                        for r, lat in zip(r_los, lat_los)])
        g_factor = thomson_geometric_factor(scatter_angle)
        # Integrate (units arbitrary)
        brightness[i] = np.sum(n_e * g_factor * SIGMA_T_CM2 * ds_cm)
    return brightness


# Build the streamer density model over the meridional plane
r_grid = np.linspace(2.0, 80.0, 40)
lat_grid = np.linspace(-60.0, 60.0, 40)
density_field = axisymmetric_streamer_density(r_grid, lat_grid)

fig, ax = plt.subplots(figsize=(9, 5))
im = ax.pcolormesh(r_grid, lat_grid, np.log10(density_field).T,
                   shading='auto', cmap='magma')
ax.set_xlabel("Heliocentric distance (R$_\odot$)")
ax.set_ylabel("Latitude (deg)")
ax.set_title("Synthetic streamer density $\log_{10}\,n_e$ (cm$^{-3}$)")
plt.colorbar(im, ax=ax, label="$\log_{10}\,n_e$ (cm$^{-3}$)")
plt.tight_layout()
plt.show()

In [ ]:
# Forward-model: compute LOS brightness from three SoloHI vantages
def density_lookup(r_rsun: np.ndarray, lat_deg: np.ndarray) -> np.ndarray:
    """Wrapper exposing the synthetic streamer density on a 2D grid."""
    return axisymmetric_streamer_density(r_rsun, lat_deg)


elongation_grid = np.linspace(6.0, 30.0, 18)
spacecraft_distance = 0.5  # AU (typical SoloHI mid-orbit)

vantages = {
    "Ecliptic (0 deg)": 0.0,
    "Out-of-ecliptic North (+30 deg)": 30.0,
    "Out-of-ecliptic South (-30 deg)": -30.0,
}

los_data = {}
for label, lat in vantages.items():
    los_data[label] = los_brightness(lat, spacecraft_distance, elongation_grid,
                                       density_lookup)

# --- Inversion: solve for radial-latitudinal density modes from the 3 vantages.
# We assume a 2-component model n_e(r, lat) = n0(r) + n1(r) * exp(-lat^2 / w^2)
# Each LOS integral becomes a linear combination of (n0, n1) at radii sampled.
# For demonstration, fit at each elongation independently.

def inversion_basis(spacecraft_distances: np.ndarray,
                     spacecraft_lats: np.ndarray,
                     elongation_deg: float) -> np.ndarray:
    """Construct the LOS->(n0, n1) basis matrix at one elongation.

    Each row is one vantage; columns are the integrated weights for the
    radial-isotropic (n0) and streamer-belt (n1) basis components.

    Args:
        spacecraft_distances: Array of spacecraft Sun-distances (AU).
        spacecraft_lats: Array of spacecraft latitudes (deg).
        elongation_deg: LOS elongation (deg).

    Returns:
        Matrix shape (n_vantages, 2).
    """
    rows = []
    for d_sc, lat_sc in zip(spacecraft_distances, spacecraft_lats):
        # Use a unit-density background model and a unit-streamer model
        bg_density = lambda r, lat: np.ones((len(r), len(lat)))  # noqa: E731
        streamer_density = lambda r, lat: np.exp(  # noqa: E731
            -0.5 * (np.meshgrid(r, lat, indexing='ij')[1] / 15.0)**2)
        b_bg = los_brightness(lat_sc, d_sc, np.array([elongation_deg]),
                               bg_density, n_los_steps=80)[0]
        b_str = los_brightness(lat_sc, d_sc, np.array([elongation_deg]),
                                 streamer_density, n_los_steps=80)[0]
        rows.append([b_bg, b_str])
    return np.array(rows)


sc_distances = np.array([spacecraft_distance, spacecraft_distance, spacecraft_distance])
sc_lats = np.array([0.0, 30.0, -30.0])

n0_recovered = []
n1_recovered = []
for eps in elongation_grid:
    A = inversion_basis(sc_distances, sc_lats, eps)
    b_obs = np.array([los_data[label][np.where(elongation_grid == eps)[0][0]]
                      for label in vantages])
    coeffs, *_ = np.linalg.lstsq(A, b_obs, rcond=None)
    n0_recovered.append(coeffs[0])
    n1_recovered.append(coeffs[1])
n0_recovered = np.array(n0_recovered)
n1_recovered = np.array(n1_recovered)

# Plot results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: brightness curves from three vantages
ax = axes[0]
for label, brightness in los_data.items():
    ax.plot(elongation_grid, brightness, marker='o', lw=1.5, label=label)
ax.set_xlabel("Solar elongation (deg)")
ax.set_ylabel(r"LOS brightness (arb. K-corona units)")
ax.set_title("Multi-vantage Thomson brightness (3 SoloHI orbits)")
ax.set_yscale('log')
ax.legend(fontsize=9)

# Right: recovered n0, n1 vs. true at corresponding r_TS
ax = axes[1]
r_ts_grid = thomson_surface_radius(elongation_grid, spacecraft_distance)
# True profiles for comparison
true_n0 = axisymmetric_streamer_density(r_ts_grid, np.array([60.0]))[:, 0]  # high lat
true_n1 = (axisymmetric_streamer_density(r_ts_grid, np.array([0.0]))[:, 0] - true_n0)
# Normalize the recovered profiles for comparison (they have arbitrary units)
n0_norm = n0_recovered / np.median(np.abs(n0_recovered))
n0_true_norm = true_n0 / np.median(np.abs(true_n0))
n1_norm = n1_recovered / np.median(np.abs(n1_recovered))
n1_true_norm = true_n1 / np.median(np.abs(true_n1))
ax.plot(r_ts_grid, n0_true_norm, 'k-', lw=2, label="True high-latitude $n_0$ (norm)")
ax.plot(r_ts_grid, n0_norm, 'C0o-', label="Recovered $n_0$ (norm)")
ax.plot(r_ts_grid, n1_true_norm, 'k--', lw=2, label="True streamer enhancement $n_1$ (norm)")
ax.plot(r_ts_grid, n1_norm, 'C3s-', label="Recovered $n_1$ (norm)")
ax.set_xlabel("Heliocentric distance r$_{TS}$ (R$_\odot$)")
ax.set_ylabel("Density component (normalised)")
ax.set_title("Tomographic inversion: 2-component density")
ax.legend(fontsize=9)
ax.set_yscale('log')

plt.tight_layout()
plt.show()

print(f"Recovered streamer enhancement is {np.mean(n1_recovered/n0_recovered):.2f}x "
      f"the background (true model: 4.0x at the equator).")

## Summary / 요약

| Concept / 개념 | This Notebook / 이 노트북 | SoloHI Reality / 실제 SoloHI |
|---|---|---|
| **APS-CMOS noise model / APS-CMOS 잡음 모델** | 5T pinned-photodiode + Poisson + read + dark + saturation, two gain modes. / 5T pinned-photodiode + Poisson + read + dark + 포화, 두 gain 모드. | High-gain 5.8 e⁻ read / low-gain 35.1 e⁻ / DR 2 463 / dark <0.3 (BOL) e⁻/px/s. / Table 1 § 4.6. |
| **Heliospheric FOV 5°–45° / 헬리오스피어 시야** | $r_{\rm TS} = d_{\rm sc}\sin\varepsilon$ projection at 0.28, 0.5, 0.88 AU. / Thomson 표면 반지름 투영. | 5.25 R☉ inner @ 0.28 AU vs HI-1's 15 R☉ at 1 AU — closer-to-Sun by ~3×. / SECCHI HI-1 대비 3배 가까이. |
| **Stray-light baffle suppression / Baffle 잡광 억제** | Fraunhofer-edge per stage + 5 forward + 9 interior + AE1/AE2 + peripheral. / 단계별 회절 + 다단 baffle 통합. | 10⁻¹⁰ B☉ (inner) → 10⁻¹³ B☉ (outer) requirement, validated in NRL SCOTCH. / Fig. 11. |
| **Out-of-ecliptic 3D Thomson reconstruction / 황도면 외 3D Thomson 재구성** | 3-vantage LOS brightness + 2-component LSQ inversion of axisymmetric streamer. / 3 시점 LOS + 2-성분 최소제곱 역산. | 30°+ Solar Orbiter inclination + 2-week N/S sweep enables direct tomography vs. LASCO single vantage. / 단일 시점 forward modelling 대비 inversion 가능. |

These four pieces — detector noise budget, FOV physical scale, baffle stray-light architecture, and multi-vantage tomography — are the analytic spine of SoloHI's instrument paper. Together they explain why SoloHI's combination of 0.28 AU perihelion + 30° ecliptic inclination + 4-die APS + multi-stage baffle system constitutes a third-generation heliospheric imager that breaks all three degrees of freedom (vantage, distance, inclination) of the LASCO/SECCHI lineage.

이 네 요소 — 검출기 잡음 예산, FOV 물리적 스케일, baffle 잡광 구조, 다중 vantage 토모그래피 — 는 SoloHI instrument paper의 분석 백본이다. 이들은 SoloHI의 0.28 AU 근일점 + 30° 황도경사 + 4-die APS + 다단 baffle 조합이 LASCO/SECCHI 계보의 세 자유도(시점·거리·경사)를 모두 깨는 3세대 헬리오스피어 영상기인 이유를 설명한다.

### References / 참고문헌

- Howard, R. A. et al., *A&A* 642, A13 (2020).
- Vourlidas, A. & Howard, R. A., *ApJ* 642, 1216 (2006).
- Howard, T. A. & Tappin, S. J., *Space Sci. Rev.* 147, 31 (2009).
- Janesick, J. et al., *Proc. SPIE* 7742, 77420B (2010).
- Korendyke, C. M. et al., *Proc. SPIE* 8862, 88620J (2013).